# Prueba de bondad de ajuste chi-cuadrado en Python

**Módulo II — Simulación de eventos discretos**

Las pruebas de bondad de ajuste permiten evaluar si las frecuencias observadas en una muestra son compatibles con una distribución teórica. En simulación se utilizan para comprobar si los datos generados presentan el comportamiento probabilístico esperado.

En este ejercicio se analizan 1000 dulces distribuidos en 10 paquetes. Cada dulce puede tener uno de cinco sabores: A, B, C, D o E. Si todos los sabores son igualmente probables, se esperan 200 dulces de cada sabor.

## Hipótesis

- **Hipótesis nula ($H_0$):** los cinco sabores aparecen en la misma proporción; cada sabor tiene probabilidad $p=0.20$.
- **Hipótesis alternativa ($H_1$):** al menos uno de los sabores presenta una proporción diferente.
- **Nivel de significancia:** $\alpha=0.05$, equivalente a un nivel de confianza del 95 %.

El estadístico de prueba es:

$$\chi^2=\sum_{i=1}^{k}\frac{(O_i-E_i)^2}{E_i},$$

donde $O_i$ es la frecuencia observada y $E_i$ es la frecuencia esperada de la categoría $i$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import exp

sabores = np.array(['A', 'B', 'C', 'D', 'E'])
observados = np.array([180, 250, 120, 225, 225])
esperados = np.full(5, observados.sum() / len(sabores))

datos = pd.DataFrame({
    'Sabor': sabores,
    'Frecuencia observada': observados,
    'Frecuencia esperada': esperados.astype(int),
})
datos

## Cálculo manual del estadístico

Se calcula la contribución de cada sabor. Una contribución grande indica una diferencia importante entre la frecuencia observada y la esperada.

In [ ]:
datos['Diferencia'] = observados - esperados
datos['Contribución chi-cuadrado'] = (observados - esperados) ** 2 / esperados
chi2_manual = datos['Contribución chi-cuadrado'].sum()
datos

In [ ]:
estadistico = chi2_manual
grados_libertad = len(sabores) - 1
alfa = 0.05

# Para 4 grados de libertad, P(Chi² >= x) = exp(-x/2) * (1 + x/2).
def probabilidad_cola_chi2_gl4(x):
    return exp(-x / 2) * (1 + x / 2)

valor_p = probabilidad_cola_chi2_gl4(estadistico)

# Búsqueda por bisección del valor crítico cuya cola es igual a alfa.
inferior, superior = 0.0, 50.0
for _ in range(100):
    medio = (inferior + superior) / 2
    if probabilidad_cola_chi2_gl4(medio) > alfa:
        inferior = medio
    else:
        superior = medio
valor_critico = (inferior + superior) / 2

print(f'Estadístico calculado manualmente: {chi2_manual:.2f}')
print(f'Estadístico chi-cuadrado: {estadistico:.2f}')
print(f'Grados de libertad: {grados_libertad}')
print(f'Valor crítico para alfa = {alfa}: {valor_critico:.4f}')
print(f'Valor p: {valor_p:.12g}')

## Regla de decisión

Se rechaza $H_0$ cuando el valor p es menor que $\alpha$. De manera equivalente, se rechaza cuando el estadístico calculado supera el valor crítico de la distribución chi-cuadrado.

In [ ]:
if valor_p < alfa:
    decision = 'Se rechaza la hipótesis nula (H₀).'
    conclusion = (
        'Las diferencias entre los sabores observados y los esperados son '
        'estadísticamente significativas. Los datos no se ajustan a una '
        'distribución uniforme entre los cinco sabores.'
    )
else:
    decision = 'No se rechaza la hipótesis nula (H₀).'
    conclusion = (
        'No se encontró evidencia estadística suficiente para afirmar que '
        'las proporciones difieren de las esperadas.'
    )

print(decision)
print(conclusion)

In [ ]:
x = np.arange(len(sabores))
ancho = 0.36
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - ancho/2, observados, ancho, label='Observados', color='#2878B5')
ax.bar(x + ancho/2, esperados, ancho, label='Esperados', color='#F5A623')
ax.set_xticks(x, sabores)
ax.set_xlabel('Sabor')
ax.set_ylabel('Número de dulces')
ax.set_title('Frecuencias observadas y esperadas por sabor')
ax.axhline(200, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.legend()
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## Interpretación de los resultados

El estadístico obtenido es $\chi^2=52.75$, con 4 grados de libertad. Este valor es muy superior al valor crítico correspondiente a un nivel de significancia de 0.05. Además, el valor p es mucho menor que 0.05. Por tanto, se rechaza la hipótesis de que los sabores se encuentran distribuidos en proporciones iguales.

La tabla de contribuciones muestra que el sabor C es el que más aporta al estadístico, porque se observaron solamente 120 dulces frente a los 200 esperados. El sabor B también presenta una diferencia importante, con 250 unidades observadas.

La expresión correcta es **rechazar la hipótesis nula**. Un valor p pequeño no significa que la hipótesis sea imposible, sino que los datos observados serían muy poco probables si la distribución uniforme propuesta fuera cierta.

## Relación con la simulación de eventos discretos

En un modelo de simulación, la prueba permite validar generadores de datos categóricos y contrastar las salidas simuladas con una distribución teórica. Puede emplearse para estudiar tipos de clientes, categorías de productos, clases de solicitudes, fallas de equipos o cualquier variable discreta representada mediante frecuencias.

## Supuestos que deben verificarse

- Las observaciones deben ser independientes.
- Las categorías deben ser mutuamente excluyentes.
- Las frecuencias esperadas deben ser suficientemente grandes; como regla práctica, se recomienda que sean al menos 5.
- La suma de las frecuencias observadas debe coincidir con la suma de las esperadas.

## Actividad propuesta

1. Identifique qué sabor realiza el mayor aporte al estadístico y explique por qué.
2. Cambie las frecuencias por `[195, 205, 190, 210, 200]` y repita la prueba.
3. Simule 1000 sabores con probabilidades iguales usando `numpy.random.choice`.
4. Compare las frecuencias simuladas con las esperadas y formule la decisión con $\alpha=0.05$.
5. Explique por qué no rechazar $H_0$ no equivale a demostrar que $H_0$ es verdadera.